# 03 — Modelling

Five model families, each tuned by cross-validation on the training set. The test set is opened once, in the final section.

## 1. Setup

### 1a. Loading the split and the preprocessors

`02_preprocessing.ipynb` saved the cleaned 80/20 split and the two preprocessor definitions. The definitions are unfitted: each one is refit on the training part of every cross-validation fold, inside the pipeline it belongs to.

In [1]:
import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
REPORTS_DIR = "../reports"

train = joblib.load(f"{PROCESSED_DIR}/train.joblib")
test = joblib.load(f"{PROCESSED_DIR}/test.joblib")

X_train, y_train = train.drop(columns="TARGET"), train["TARGET"]
X_test, y_test = test.drop(columns="TARGET"), test["TARGET"]

preprocessor_linear = joblib.load(f"{MODELS_DIR}/preprocessor_linear.joblib")
preprocessor_tree = joblib.load(f"{MODELS_DIR}/preprocessor_tree.joblib")

for name, X_part, y_part in [("train", X_train, y_train), ("test", X_test, y_test)]:
    print(f"{name:<6}{len(X_part):>8,} rows   {X_part.shape[1]:>3} features   "
          f"default rate {y_part.mean():.4f}")

train  246,003 rows    60 features   default rate 0.0807
test    61,501 rows    60 features   default rate 0.0807


### 1b. Cross-validation

One `StratifiedKFold` object, reused by every search below, so each family is scored on identical folds and the means are directly comparable. Both metrics are computed per fold; PR-AUC decides which configuration wins.

In [2]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

SCORING = {"ap": "average_precision", "roc": "roc_auc"}
REFIT = "ap"

held_out = len(next(iter(cv.split(X_train, y_train)))[1])
print(f"{N_SPLITS}-fold stratified cross-validation on the training set")
print(f"  fit on   {len(X_train) - held_out:,} rows per fold")
print(f"  score on {held_out:,} rows per fold")
print(f"\nno-skill floors   PR-AUC {y_train.mean():.4f} (the positive rate)   ROC-AUC 0.5000")

5-fold stratified cross-validation on the training set
  fit on   196,802 rows per fold
  score on 49,201 rows per fold

no-skill floors   PR-AUC 0.0807 (the positive rate)   ROC-AUC 0.5000


### 1c. Pipeline shapes

Three wrappers, one per representation. Each joins a preprocessor to a model in a single estimator, which is what lets the search refit the preprocessor on every fold.

XGBoost and LightGBM read a pandas `category` dtype rather than integer codes, so a cast step sits between the two.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()


def as_categorical(X):
    """Integer codes -> pandas category dtype. For XGBoost and LightGBM only."""
    X = X.copy()
    for col in X.columns:
        if col.startswith(("categorical__", "ordinal__")):
            X[col] = X[col].astype("int8").astype("category")
    return X


def linear_pipe(model):
    """Logistic Regression: median impute, scale, one-hot."""
    return Pipeline([("prep", preprocessor_linear), ("model", model)])


def tree_pipe(model):
    """Decision Tree, Random Forest: integer codes, NaN kept."""
    return Pipeline([("prep", preprocessor_tree), ("model", model)])


def boost_pipe(model):
    """XGBoost, LightGBM: integer codes recast as pandas category."""
    return Pipeline([
        ("prep", preprocessor_tree),
        ("cast", FunctionTransformer(as_categorical)),
        ("model", model),
    ])


print(f"scale_pos_weight = {SCALE_POS_WEIGHT:.3f}")

scale_pos_weight = 11.387


### 1d. The search helper

`scoring`, `refit`, `cv` and `return_train_score` are set in one place, so no family is evaluated on different terms from another. `n_iter=None` runs an exhaustive grid, any other value samples that many configurations.

Each call records one row: the winning configuration's cross-validated mean and standard deviation, its training score, and what the search cost.

In [5]:
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

results = []


def run_search(name, pipe, space, n_iter=None, n_jobs=6):
    """Tune one family on the training set. n_iter=None -> exhaustive grid."""
    shared = dict(scoring=SCORING, refit=REFIT, cv=cv, return_train_score=True, n_jobs=n_jobs)
    if n_iter is None:
        search = GridSearchCV(pipe, space, **shared)
    else:
        search = RandomizedSearchCV(pipe, space, n_iter=n_iter,
                                    random_state=RANDOM_STATE, **shared)

    start = time.perf_counter()
    search.fit(X_train, y_train)
    wall = time.perf_counter() - start

    cv_res, best = search.cv_results_, search.best_index_
    row = {
        "model": name,
        "cv_ap": cv_res["mean_test_ap"][best],
        "cv_ap_std": cv_res["std_test_ap"][best],
        "cv_roc": cv_res["mean_test_roc"][best],
        "train_ap": cv_res["mean_train_ap"][best],
        "candidates": len(cv_res["params"]),
        "wall_min": wall / 60,
        "best_params": search.best_params_,
    }
    results[:] = [r for r in results if r["model"] != name] + [row]

    print(f"{name}")
    print(f"  CV PR-AUC    {row['cv_ap']:.4f} +/- {row['cv_ap_std']:.4f}")
    print(f"  CV ROC-AUC   {row['cv_roc']:.4f}")
    print(f"  train PR-AUC {row['train_ap']:.4f}   gap {row['train_ap'] - row['cv_ap']:+.4f}")
    print(f"  {row['candidates']} candidates x {N_SPLITS} folds in {row['wall_min']:.1f} min")
    for key, value in search.best_params_.items():
        print(f"    {key.replace('model__', '')} = {value}")
    return search

## 2. Logistic Regression

One grid over the strength of the penalty (`C`) and its composition (`l1_ratio`) — 0 is pure L2, 1 is pure L1, 0.5 an even elastic-net mix. The `saga` solver handles all three, so the `C` values stay comparable across the row. `tol=1e-2`, which saga reaches in about 18 iterations.

9 configurations x 5 folds.

In [11]:
from sklearn.linear_model import LogisticRegression

logreg_search = run_search(
    "Logistic Regression",
    linear_pipe(LogisticRegression(
        solver="saga", 
        max_iter=300,
        tol=1e-2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
    {
        "model__C": [0.01, 0.1, 1.0],
        "model__l1_ratio": [0.0, 0.5, 1.0],
    },
)

Logistic Regression
  CV PR-AUC    0.2206 +/- 0.0037
  CV ROC-AUC   0.7449
  train PR-AUC 0.2214   gap +0.0008
  9 candidates x 5 folds in 2.5 min
    C = 0.1
    l1_ratio = 1.0


### 2a. Coefficients of the winning configuration

The numeric branch is standardised, so each coefficient is the change in log-odds of default per one standard deviation of that feature, and `exp(coefficient)` is the multiplicative change in the odds.

In [13]:
best_logreg = logreg_search.best_estimator_
feature_names = best_logreg.named_steps["prep"].get_feature_names_out()
coefficients = pd.Series(best_logreg.named_steps["model"].coef_[0], index=feature_names)

print(f"{len(coefficients)} features in   "
      f"{int((coefficients == 0).sum())} zeroed by the penalty\n")

top = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index).head(15)
print(pd.DataFrame({"coefficient": top, "odds_ratio": np.exp(top)}).to_string(
    formatters={"coefficient": "{:+.3f}".format, "odds_ratio": "{:.3f}".format}))

96 features in   2 zeroed by the penalty

                                                 coefficient odds_ratio
numeric__EXT_SOURCE_3                                 -0.488      0.614
numeric__EXT_SOURCE_2                                 -0.391      0.676
categorical__CODE_GENDER_M                            +0.318      1.374
categorical__FLAG_OWN_CAR_Y                           -0.237      0.789
categorical__NAME_CONTRACT_TYPE_Revolving loans       -0.229      0.796
numeric__EXT_SOURCE_1                                 -0.203      0.817
categorical__NAME_FAMILY_STATUS_Married               -0.164      0.849
numeric__EMPLOYMENT_YEARS                             -0.148      0.862
ordinal__NAME_EDUCATION_TYPE                          -0.146      0.864
flags__FLAG_DOCUMENT_3                                +0.140      1.150
numeric__IS_MISSING_EXT_SOURCE_1                      +0.102      1.108
categorical__NAME_INCOME_TYPE_Pensioner               -0.101      0.904
numeric__IS_MISSING_AM